In [0]:
"""
Load the CSV containing the map from search term to note group into a dataframe
"""

search_criteria_map_file='/Workspace/Users/dax.westerman@vumc.org/tedla-hypertension-project/v0.0.2/raw_data/extraction_details.csv'

import pandas as pd

search_criteria_map_df = pd.read_csv(search_criteria_map_file)

search_criteria_map_df.head()


,search_term_to_note_groups_id,search_term,admission note,communication encounter (phone and my health note),discharge summary,ecg impression,emergency department note,inpatient note,outpatient note,problem list
0,0,heart failure with reduced ejection fraction,1,0,1,1,1,1,1,1
1,1,hfref,1,0,1,1,1,1,1,1
2,2,heart failure with preserved ejection fraction,1,0,1,1,1,1,1,1
3,3,hfpef,1,0,1,1,1,1,1,1
4,4,hyponatremia,1,1,1,0,1,1,1,1


In [0]:
"""
Clean up column names before loading into database
"""
import re

def clean_col(col):
    col = re.sub(r"\s+", "_", col)
    col = re.sub(r"\([^)]*\)", "", col)
    return col.strip("_")

search_criteria_map_df.columns = [clean_col(c) for c in search_criteria_map_df.columns]
if 'search_term_to_note_groups_id' in search_criteria_map_df.columns:
    search_criteria_map_df['search_term_to_note_groups_id'] = range(1, len(search_criteria_map_df) + 1)

display(pd.DataFrame({'columns': search_criteria_map_df.columns}))


columns
search_term_to_note_groups_id
search_term
admission_note
communication_encounter
discharge_summary
ecg_impression
emergency_department_note
inpatient_note
outpatient_note
problem_list


In [0]:
"""
Sample
"""
search_criteria_map_df.head()

,search_term_to_note_groups_id,search_term,admission_note,communication_encounter,discharge_summary,ecg_impression,emergency_department_note,inpatient_note,outpatient_note,problem_list
0,1,heart failure with reduced ejection fraction,1,0,1,1,1,1,1,1
1,2,hfref,1,0,1,1,1,1,1,1
2,3,heart failure with preserved ejection fraction,1,0,1,1,1,1,1,1
3,4,hfpef,1,0,1,1,1,1,1,1
4,5,hyponatremia,1,1,1,0,1,1,1,1


In [0]:
"""
Create table for map between search term and associated note groups
"""
dest_table = "default.map_search_term_to_note_group"
spark.sql(f"DROP TABLE IF EXISTS {dest_table}")
spark_df = spark.createDataFrame(search_criteria_map_df)
spark_df = spark_df.withColumnRenamed("admission_note", "notes_admissions") \
    .withColumnRenamed("communication_encounter", "notes_communication_encounter") \
    .withColumnRenamed("discharge_summary", "notes_discharge_summary") \
    .withColumnRenamed("ecg_impression", "notes_ecg_impression") \
    .withColumnRenamed("emergency_department_note", "notes_emergency_department") \
    .withColumnRenamed("inpatient_note", "notes_inpatient") \
    .withColumnRenamed("outpatient_note", "notes_outpatient") \
    .withColumnRenamed("problem_list", "notes_problem_lists")
spark_df.write.mode("overwrite").saveAsTable(dest_table)

In [0]:
df_loaded = spark.read.table(dest_table)
display(df_loaded)

search_term_to_note_groups_id,search_term,notes_admissions,notes_communication_encounter,notes_discharge_summary,notes_ecg_impression,notes_emergency_department,notes_inpatient,notes_outpatient,notes_problem_lists
1,heart failure with reduced ejection fraction,1,0,1,1,1,1,1,1
2,hfref,1,0,1,1,1,1,1,1
3,heart failure with preserved ejection fraction,1,0,1,1,1,1,1,1
4,hfpef,1,0,1,1,1,1,1,1
5,hyponatremia,1,1,1,0,1,1,1,1
6,low sodium,1,1,1,0,1,1,1,1
7,low na,1,1,1,0,1,1,1,1
8,hypokalemia,1,1,1,0,1,1,1,1
9,lower k,1,1,1,0,1,1,1,1
10,hypo k,1,1,1,0,1,1,1,1



Location of query to check data

```
/Workspace/Users/dax.westerman@vumc.org/tedla-hypertension-project/v0.0.2/note_search_method.dbquery.ipynb
```

